# Data Preparation

This notebook prepares the California Housing Prices dataset for machine learning.

The preparation process includes splitting the data into training and test sets, handling missing values, engineering useful features, encoding categorical variables, scaling numerical features when appropriate, and building a preprocessing pipeline.

All preprocessing steps will be fitted using the training data to prevent data leakage.

## 1. Import Libraries

Import the libraries required for data preparation.

In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

## 2. Load Dataset

Load the housing dataset into a pandas DataFrame.

In [16]:
df = pd.read_csv("../data/housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## 3. Separate Features and Target

The target variable, `median_house_value`, is separated from the input features to prepare the dataset for supervised learning.

In [17]:
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (20640, 9)
Target shape: (20640,)


## 4. Split the Dataset

The dataset is split into training and test sets to evaluate how well the model generalizes to unseen data.

The training set will be used to fit the model and learn preprocessing parameters, while the test set will be reserved for final model evaluation.

An 80/20 split is used, with a fixed random state to ensure reproducibility.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (16512, 9)
X_test: (4128, 9)
y_train: (16512,)
y_test: (4128,)


## 5. Identify Numerical and Categorical Features

The input features contain both numerical and categorical variables. Identifying their data types allows us to apply appropriate preprocessing techniques to each group.

Numerical features may require missing-value imputation and scaling, while categorical features require encoding before being used by most machine learning models.

In [19]:
numerical_features = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
Categorical features: ['ocean_proximity']


## 6. Feature Engineering

New features are created from existing numerical features to provide more meaningful information to the model. In this dataset, ratios such as rooms per household, bedrooms per room, and population per household may capture useful characteristics of each district.

The same transformations are applied to both training and test sets. The target variable is not used to create these features.

In [20]:
for data in [X_train, X_test]:
    data["rooms_per_household"] = (
        data["total_rooms"] / data["households"]
    )

    data["bedrooms_per_room"] = (
        data["total_bedrooms"] / data["total_rooms"]
    )

    data["population_per_household"] = (
        data["population"] / data["households"]
    )

In [21]:
X_train[
    [
        "rooms_per_household",
        "bedrooms_per_room",
        "population_per_household"
    ]
].head()

,rooms_per_household,bedrooms_per_room,population_per_household
14196,5.017657,0.200576,3.691814
8267,4.473545,0.232703,1.738095
17445,5.645833,0.174486,2.723214
14265,4.002817,0.258269,3.994366
2271,6.268421,0.180940,2.300000


In [22]:
numerical_features = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("Number of numerical features:", len(numerical_features))
print("Numerical features:", numerical_features)

print("Number of categorical features:", len(categorical_features))
print("Categorical features:", categorical_features)

Number of numerical features: 11
Numerical features: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household']
Number of categorical features: 1
Categorical features: ['ocean_proximity']


### Observations

- The `rooms_per_household` values indicate the average number of rooms per household. For the first five displayed records, the values range from approximately 4.00 to 6.27 rooms per household.
- The `bedrooms_per_room` values range from approximately 0.17 to 0.26 in these records, meaning that bedrooms account for roughly 17%–26% of the total rooms.
- The `population_per_household` values range from approximately 1.74 to 3.99 people per household in these records.
- These ratios provide a more interpretable view of housing characteristics and household composition than raw counts alone.
- These observations are based only on the first five displayed records and should not be generalized to the entire dataset without further analysis.
- The predictive value of these engineered features will be assessed during model evaluation.

## 7. Handling Pre-existing Capping

### Observations

During data understanding, two variables were found to have values concentrated at their respective upper limits:

- `median_house_value`: 965 records (approximately 4.68%) have a value of 500,001.
- `housing_median_age`: 1,273 records (approximately 6.17%) have a value of 52.

These patterns suggest that the original dataset may have capped these values. Therefore, the recorded values may not represent the true values beyond those limits.

### Decision

No additional capping will be applied to these variables at this stage.

The existing limits will be documented as a dataset limitation. In particular, the cap on `median_house_value` may limit the model's ability to distinguish between districts whose actual median house values exceed 500,001.

The effect of this limitation will be considered when interpreting model predictions and evaluation results.

## 8. Build the Preprocessing Pipeline

A preprocessing pipeline applies a consistent sequence of transformations to the data.

Numerical features are imputed using the median and scaled with StandardScaler. Categorical features are imputed using the most frequent value and encoded using OneHotEncoder.

A ColumnTransformer applies the appropriate pipeline to each feature group. The transformers will be fitted on the training data only, helping prevent data leakage.

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [24]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

## 9. Fit and Apply the Preprocessor

In this step, the preprocessing pipeline is fitted on the training data and then applied to both the training and test datasets.

During fitting, the pipeline learns the median values for imputing missing numerical data, the most frequent category for imputing missing categorical data, the scaling parameters for numerical features, and the categories used for one-hot encoding.

The fitted pipeline is then used to transform both datasets. The test data is transformed using parameters learned exclusively from the training data, preventing data leakage.

Finally, the shapes of the prepared datasets are checked to verify that both datasets have the expected number of records and features.

In [25]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Prepared training data shape:", X_train_prepared.shape)
print("Prepared test data shape:", X_test_prepared.shape)

Prepared training data shape: (16512, 16)
Prepared test data shape: (4128, 16)


### Observations

- The preprocessing pipeline successfully transformed both datasets.
- The training set contains 16,512 records, and the test set contains 4,128 records.
- Both datasets contain 16 features after preprocessing.
- All 11 numerical features (the 8 original ones and the 3 engineered ratios) were retained, while the categorical feature `ocean_proximity` was converted into five columns through one-hot encoding.
- The number of records remained unchanged during preprocessing.

**Conclusion:** The preprocessing pipeline has been applied successfully to both the training and test datasets. The prepared datasets are ready for validation before model training.

## 10. Validate the Prepared Data

Before training a machine learning model, the prepared datasets should be checked for invalid numerical values.

Feature engineering and preprocessing can sometimes introduce missing values (`NaN`) or infinite values (`inf`), particularly when mathematical operations involve division.

This validation checks whether the prepared training data contains any `NaN` or infinite values. These checks help identify potential issues that could interfere with model training.

The same validation will also be performed on the test data to ensure that both datasets are ready for the next stage.

In [26]:
import scipy.sparse as sp

for name, data in [
    ("Training", X_train_prepared),
    ("Test", X_test_prepared)
]:
    if sp.issparse(data):
        data = data.data

    print(f"{name} data contains NaN:", np.isnan(data).any())
    print(f"{name} data contains infinity:", np.isinf(data).any())

Training data contains NaN: False
Training data contains infinity: False
Test data contains NaN: False
Test data contains infinity: False


### Validation Results

- No `NaN` values were found in the prepared training or test data.
- No infinite values were found in either dataset.
- The validation confirms that the preprocessing pipeline successfully handled missing values and produced finite numerical features.

**Conclusion:** The prepared datasets passed these validation checks and are ready for model training. This validation does not guarantee that the data is free from every possible issue, but it confirms that no missing or infinite values were detected in the prepared features.

## 11. Save Prepared Data and Preprocessor

The prepared training and test datasets, target variables, and fitted preprocessing pipeline are saved for reuse in subsequent modeling experiments.

Saving these artifacts avoids repeating the preprocessing steps and ensures that future models use the same data split and preprocessing parameters.

The fitted preprocessor will also be retained so that new data can be transformed consistently before making predictions.

In [27]:
import os
import joblib

artifacts_dir = "../artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

joblib.dump(X_train_prepared, os.path.join(artifacts_dir, "X_train_prepared.joblib"))
joblib.dump(X_test_prepared, os.path.join(artifacts_dir, "X_test_prepared.joblib"))
joblib.dump(y_train, os.path.join(artifacts_dir, "y_train.joblib"))
joblib.dump(y_test, os.path.join(artifacts_dir, "y_test.joblib"))
joblib.dump(preprocessor, os.path.join(artifacts_dir, "preprocessor.joblib"))

['../artifacts\\preprocessor.joblib']

In [28]:
import os

print(os.listdir("../artifacts"))

['best_model.joblib', 'best_model_name.joblib', 'cv_results.joblib', 'preprocessor.joblib', 'test_predictions.joblib', 'X_test_prepared.joblib', 'X_train_prepared.joblib', 'y_test.joblib', 'y_train.joblib']
